# Sprawozdanie
## Environment Initialization

In [ ]:
from os import getenv
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

In [ ]:
GOOGLE_DRIVE_PROJECT_DIRECTORY="./drive/MyDrive/Solvro/4"
ENVIRONMENTS = ['Local', 'Colab']
environment = 'Local'
if environment not in ENVIRONMENTS:
    print(f"Invalid environment detected: \"{environment}\"!")
elif environment == 'Local':
    print(f"Path set up correctly for environment: \"{environment}\"!")
elif environment == 'Colab':
    from google.colab import drive
    from pathlib import Path
    !pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers chromadb langchain-chroma pymupdf opentelemetry-api opentelemetry-sdk
    !pip install langchain-google-genai
    drive.mount('/content/drive')
    base_path = Path(GOOGLE_DRIVE_PROJECT_DIRECTORY,)
    %cd {GOOGLE_DRIVE_PROJECT_DIRECTORY}
    !ls
    print(f"Path set up correctly for environment: \"{environment}\"!")

CHUNK_SIZE=200
CHUNK_OVERLAP=50


## API Key
### Przygotowanie API key
Do stworzenia agenta użyję model językowy przechowywany w chmurzę. Każde zapytanie do agenta będzie przesyłane przez internet do providera. Formą uwierzytelnienia mnie z serverem będzie personalny klucz nazywany **api key**'em.
Mój api key zapiszę w pliku `.env`. Dodaję `.env` do `.gitignore` by nikt inny się o nim nie dowiedział.

In [ ]:
# !echo "GEMINI_API_KEY=<mój-api-key-do-modeli-od-google-blablabla>"
# !echo ".env" >> .gitignore

### Ładowanie API key
Potrzebuje teraz wczytać klucz do programu.

In [ ]:
def load_api_key(environment_variable_name):
    _ = load_dotenv()
    api_key = getenv(environment_variable_name)
    if not api_key:
        raise ValueError(f"No key found: \"{environment_variable_name}\"!")
    return api_key

API_KEY = load_api_key("GEMINI_API_KEY")
print(f"Key loaded: \"GEMINI_API_KEY\"!")

## Dokumenty PDF
Z tytułu, iż jestem osamotnionym rozbitkiem na bezludnej wyspie, najbardziej interesuje mnie jedna następująca kwestia - jakie dania jestem w stanie przygotować z kokosów.
### Ładowanie PDFów do programu

In [ ]:
def load_pdf(pdf_path : str):
    loader = PyMuPDFLoader(file_path=pdf_path)
    return loader.load()

def load_pdfs(pdf_paths):
    pdfs_loaded = list()
    total_pages_loaded = 0
    for pdf_path in pdf_paths:
        pdf_loaded = load_pdf(pdf_path)
        total_pages_loaded += len(pdf_loaded)
        pdfs_loaded.extend(pdf_loaded)
    return total_pages_loaded, pdfs_loaded


pdf_paths = [
    './pdfy/babeczki_kokosowe.pdf', './pdfy/kokosanki.pdf',
    './pdfy/lody_kokosowe.pdf',
    './pdfy/kokosanka.pdf',
    './pdfy/likier_kokosowy.pdf'
]

total_pages_loaded, pdfs_loaded = load_pdfs(pdf_paths)
print(f"Total loaded pages: {total_pages_loaded}")

### Chunkowanie 
W celach czysto optymalizacyjnych, nie chcemy LLM-owi wysyłać wszystkich pdfów na raz, gdyż do utworzenia odpowiedzi model często będzie potrzebował kilku fragmentów. Po to jest nam chunking - dzielenie załadowanych pdfów na mniejsze fragmenty.

In [ ]:
def chunk_pdfs(loaded_pdfs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    docs = text_splitter.split_documents(pdfs_loaded)
    return docs
docs = chunk_pdfs(pdfs_loaded)
print("Loaded chunks:", len(docs))

### Utworzenie Bazy Wektorowej
Fajnie, że zbiór danych jest po-chunkowany. Ale żeby optymalizacja miała sens, musimy mniej więcej umieć przewidzieć, które fragmenty dokumentów będą potrzebne LLMowi do odpowiedzi (w przeciwnym razie biędzie trzeba wysłać wszystkie chunki!). Użyjemy tutaj mechanizmu generacji embeddingów. 
Embeddingi to numeryczna reprezentacja zawartości chunka. Zapytanie użytkownika także przetłumaczymy na wartość numeryczną i dzięki porównaniu wartości embeddingów jesteśmy w stanie numerycznie wyznaczych top X chunków które poruszają tematykę zawartą w pytaniu. W tym celu stworzymy bazę wektorową.

In [ ]:
def create_vector_store(docs):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        # persist_directory="/content/drive/MyDrive/Colab Notebooks/ZadanieAgenty/"
        )
    return vectorstore

vector_store = create_vector_store(docs)

## Agent
